# Lesson 7 : Harness agent in MAF

Microsoft Agent Framework provides fully-featured pre-built agent template for production patterns, called harness agent - in which the following features are assembled by default. :

- Default harness instruction out of the box
- Hosted web search tool out of the box 
- ```InMemoryHistoryProvider``` for the memory history
- ```ContextWindowCompactionStrategy``` in before-strategy and ```ToolResultCompactionStrategy``` in after-strategy
- ```TodoProvider``` for todo list management
- ```AgentModeProvider``` for plan/execute mode tracking

You can also add features or customize existing features if required.<br>
For instance, the following is the optional settings in harness agent.

- Function invocation, when ```tools``` property is provided.
- ```MemoryContextProvider``` for persistent file-based memory, when ```memory_store``` property is provided. (Instantiated by ```MemoryContextProvider(store=memory_store)```.)
- ```SkillsProvider``` for agent skills, when ```skills_provider``` or ```skills_paths``` property is provided.

> Note : For the latest detailed implementation, please refer to the source code of ```create_harness_agent()```.

## Harness Agent (out of the box)

In the first example, we explore harness agent with default settings.

Before staring, same as in Lesson 1, we create a client as follows to run on Microsoft Foundry.

In [1]:
from dotenv import load_dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity.aio import AzureCliCredential

load_dotenv()

credential = AzureCliCredential()
client = FoundryChatClient(credential=credential)

Harness agent is instantiated by ```create_harness_agent()```.<br>
In the following code, we create a harness agent with minimal settings (without any additional settings).

In the default harness instruction, it tells to break the work into clear steps.<br>
For this reason, the agent returns the response which tells the user to clarify instructions, as shown below. (Please compare with the response in Lesson 4.)

> Note : The harness instruction can be modified (customized) by specifying ```harness_instructions``` property. If you want to maintain the harness instruction and set additional instructions, specify ```agent_instructions``` property.

In [2]:
from agent_framework import create_harness_agent

agent = create_harness_agent(
    client=client,
    name="HarnessAgent",
    max_context_window_tokens=128000,
    max_output_tokens=16384,
)

In [3]:
session = agent.create_session()
result = await agent.run(
    "Tell me the weather in Osaka today.",
    session=session,
)
print(result.text)

I can tell you, but I need one quick detail: when you say “today,” do you mean **today in Osaka (Japan time, JST)** or **today in your local time (U.S.)**?

- **Osaka today (JST): Friday, June 5, 2026**
- **U.S. today:** Friday, June 5, 2026 (but the weather “day” in Osaka may not line up with your local day)

Reply with **“Osaka time”** or **“my time”** and I’ll give the forecast (high/low, rain chance, wind, and any alerts).


In [4]:
result = await agent.run(
    "Osaka time",
    session=session,
)
print(result.text)

**Osaka weather for today (Osaka time/JST): Friday, June 5, 2026**

- **Outlook:** Rain early, becoming cloudy later (“RAIN, CLOUDY LATER”). ([data.jma.go.jp](https://www.data.jma.go.jp/multi/yoho/yoho_detail.html?code=270000&lang=en))  
- **Temperature:** **High 26°C / Low 19°C**. ([data.jma.go.jp](https://www.data.jma.go.jp/multi/yoho/yoho_detail.html?code=270000&lang=en))  
- **Chance of precipitation (JMA time blocks):**
  - **00–06:** 50%
  - **06–12:** 50%
  - **12–18:** 20%
  - **18–24:** 20% ([data.jma.go.jp](https://www.data.jma.go.jp/multi/yoho/yoho_detail.html?code=270000&lang=en))  

If you want, tell me what time range you’ll be out (morning/afternoon/evening) and I’ll translate that into a quick “umbrella or not” suggestion.


## Harness agent with custom settings (Tool calling example)

In the next example, we change harness agent to use only local functiosn instead of using web search tool, as we did in the previous examples.

In the following code, we supress web search tool by specifying ```disable_web_search=True```, and set local functions by specifying ```tools``` property.

For available properties, see the implementation of ```create_harness_agent()```.

In [6]:
from agent_framework import tool
from typing import Annotated
from pydantic import Field
from random import randint

@tool(approval_mode="never_require")
def get_weather(
    location: Annotated[str, Field(description="the location to get the weather for")],
) -> str:
    """Get the weather for a given location."""
    conditions = ["sunny", "cloudy", "rainy", "stormy"]
    return f"The weather in {location} is {conditions[randint(0, 3)]}."

@tool(approval_mode="never_require")
def get_temperature(
    location: Annotated[str, Field(description="the location to get the temperature for")],
) -> str:
    """Get the temperature for a given location."""
    return f"The temperature in {location} is {randint(10, 30)} degrees."

In [7]:
agent = create_harness_agent(
    client=client,
    name="HarnessAgent",
    max_context_window_tokens=128000,
    max_output_tokens=16384,
    disable_web_search=True,
    tools=[get_weather, get_temperature],
)

Now let's invoke the harness agent as follows.

In [8]:
session = agent.create_session()
result = await agent.run(
    "Tell me the weather in Osaka.",
    session=session,
)
print(result.text)

The weather in Osaka is **rainy**.
